# Task 4: Conversational Interface - Chat with Llama 3 and Titan Premier LLMs

In this notebook, you build a chatbot using the llama3-8b-instruct and titan-text-premier Foundation Models (FMs) in Amazon Bedrock.

Conversational interfaces such as chatbots and virtual assistants can enhance the user experience for your customers. Chatbots use natural language processing (NLP) and machine learning algorithms to understand and respond to user queries. You can use chatbots in a variety of applications, such as customer service, sales, and e-commerce, to provide quick and efficient responses to users. Users can access them through various channels such as websites, social media platforms, and messaging apps.

- **Chatbot (Basic)**, Zero Shot chatbot with a FM model
- **Chatbot using prompt**, template(LangChain) - Chatbot with some context provided in the prompt template
- **Chatbot with persona**, Chatbot with defined roles. i.e. Career Coach and Human interactions
- **Contextual-aware chatbot**, Passing in context through an external file by generating embeddings.

## LangChain framework for building Chatbot with Amazon Bedrock

In conversational interfaces such as chatbots, remembering previous interactions becomes highly important, both at a short-term and long-term level.

The LangChain framework provides memory components in two forms. First, LangChain provides helper utilities for managing and manipulating previous chat messages. These are designed to be modular. Secondly, LangChain provides easy ways to incorporate these utilities into chains, allowing you to easily define and interact with different types of abstractions, which make powerful chatbots easy for you to build.

## Building a Chatbot with Context - Key Elements

The first process in building a context-aware chatbot is to generate embeddings for the context. Typically, you have an ingestion process which runs through your embedding model and generates the embeddings, which will then be stored in a vector store. In this notebook, you use the Titan Embeddings model for this. The second process is user request orchestration, interaction, invoking, and returning the results. This involves orchestrating the user request, interacting with the necessary models/components to gather information, invoking the chatbot to formulate a response, and then returning the chatbot's response back to the user.

## Task 4.1: Environment setup

In this task, you set up your environment.

In [1]:
#ignore warnings and create a service client by name using the default session.
import json
import os
import sys
import warnings

import boto3

warnings.filterwarnings('ignore')
module_path = ".."
sys.path.append(os.path.abspath(module_path))
bedrock_client = boto3.client('bedrock-runtime',region_name=os.environ.get("AWS_DEFAULT_REGION", None))


In [2]:
# format instructions into a conversational prompt
from typing import Dict, List

def format_instructions(instructions: List[Dict[str, str]]) -> List[str]:
    """Format instructions where conversation roles must alternate system/user/assistant/user/assistant/..."""
    prompt: List[str] = []
    for instruction in instructions:
        if instruction["role"] == "system":
            prompt.extend(["<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n", (instruction["content"]).strip(), " <|eot_id|>"])
        elif instruction["role"] == "user":
            prompt.extend(["<|start_header_id|>user<|end_header_id|>\n", (instruction["content"]).strip(), " <|eot_id|>"])
        else:
            raise ValueError(f"Invalid role: {instruction['role']}. Role must be either 'user' or 'system'.")
    prompt.extend(["<|start_header_id|>assistant<|end_header_id|>\n"])
    return "".join(prompt)

## Task 4.2: Using chat history from LangChain to start the conversation

In this task, you enable the chatbot to carry conversational context across multiple interactions with users. Having a conversational memory is crucial for Chatbots to hold meaningful, coherent dialogues over time.

You implement conversational memory capabilities by building on top of LangChain's InMemoryChatMessageHistory class. This object stores the conversations between the user and the chatbot, and the history is available the chatbot agent so that it can leverage the context from a previous conversation.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** The model outputs are non-deterministic.

In [13]:
import boto3

bedrock_control = boto3.client(
    "bedrock",
    region_name="us-east-2"
)

profiles = bedrock_control.list_inference_profiles()

for profile in profiles["inferenceProfileSummaries"]:
    print("Profile ID:", profile["inferenceProfileId"])
    print("Status:", profile["status"])

Profile ID: us.anthropic.claude-3-haiku-20240307-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-2-1b-instruct-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-2-11b-instruct-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-2-3b-instruct-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-2-90b-instruct-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-1-8b-instruct-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-1-70b-instruct-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-1-405b-instruct-v1:0
Status: ACTIVE
Profile ID: us.meta.llama3-3-70b-instruct-v1:0
Status: ACTIVE
Profile ID: us.deepseek.r1-v1:0
Status: ACTIVE
Profile ID: us.mistral.pixtral-large-2502-v1:0
Status: ACTIVE
Profile ID: us.amazon.nova-premier-v1:0
Status: ACTIVE
Profile ID: us.stability.stable-conservative-upscale-v1:0
Status: ACTIVE
Profile ID: us.stability.stable-creative-upscale-v1:0
Status: ACTIVE
Profile ID: us.stability.stable-fast-upscale-v1:0
Status: ACTIVE
Profile ID: us.stability.stable-image-control-sketch-v1:0
Status: 

In [32]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_aws import ChatBedrock

chat_model = ChatBedrock(
    model_id="us.amazon.nova-2-lite-v1:0",
    client=bedrock_client
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the following questions as best you can."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

history = InMemoryChatMessageHistory()

def get_history():
    return history

chain = prompt | chat_model | StrOutputParser()

wrapped_chain = RunnableWithMessageHistory(
    chain,
    get_history,
    history_messages_key="chat_history",
)

query = "tell me few things about gardening.?"

response = wrapped_chain.invoke(
    {"input": query}
)

print(response)
print(history)

/opt/conda/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3579: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### **Few Things About Gardening**

Gardening is a rewarding and fulfilling hobby that connects people with nature, provides fresh produce, and enhances the beauty of living spaces. Here are **key aspects and tips** to get you started or improve your gardening skills:

---

## **1. Choose the Right Plants for Your Climate**
- **Know Your Hardiness Zone:** Use the USDA Plant Hardiness Zone Map to determine which plants, flowers, and vegetables can thrive in your area based on winter temperatures.
- **Consider Microclimates:** Even within a zone, factors like sunlight, wind, and soil type can create microclimates in your garden.
- **Native Plants:** They are adapted to local conditions, require less maintenance, and support local wildlife.

---

## **2. Soil is the Foundation**
- **Test Your Soil:** Check pH levels and nutrient content. Most vegetables prefer a pH between 6.0 and 7.0.
- **Improve Soil Health:**
  - Add **compost** or **organic matter** to increase fertility and drainage.

### New Questions

The model has responded with an initial message. Now, you ask it a few questions.

In [33]:
#new questions
instructions = [{"role": "user", "content": "Give me a few tips on how to get concentrated."}]
response=wrapped_chain.invoke({"input": format_instructions(instructions)})
print(response)

### Tips to Improve Focus and Concentration

Staying concentrated can be challenging, especially in a world filled with distractions. Here are **practical and effective tips** to help you improve your focus and stay on task:

---

## **1. Set Clear Goals**
- **Break Tasks into Smaller Steps:** Large tasks can feel overwhelming and reduce motivation. Divide them into manageable chunks with specific, achievable goals.
- **Use the SMART Criteria:** Make your goals **Specific, Measurable, Achievable, Relevant, and Time-bound**.
- **Prioritize Tasks:** Use methods like the **Eisenhower Matrix** or **To-Do Lists** to focus on what truly matters.

---

## **2. Eliminate Distractions**
- **Create a Focused Environment:**
  - Find a quiet, organized workspace.
  - Turn off notifications on your phone or computer (use “Do Not Disturb” mode).
  - Use noise-canceling headphones or calming background music (e.g., instrumental or focus playlists).
- **Declutter Your Space:** A tidy workspace helps a

### Build on the questions

Now, ask a question without mentioning the word garden to see if the model can understand the previous conversation.

In [36]:
# build on the questions
instructions = [{"role": "user", "content": "focus"}]
response=wrapped_chain.invoke({"input": format_instructions(instructions)})
print(response)

### Enhancing Focus: Practical Strategies for Better Concentration

Focus is a skill that can be developed and strengthened over time. Whether you're tackling work, studying, or trying to enjoy a hobby, here are **proven techniques** to help you stay sharp and focused:

---

## **1. Optimize Your Environment**
- **Minimize Distractions:**
  - Turn off phone notifications.
  - Use apps like **Freedom**, **Cold Turkey**, or **StayFocusd** to block distracting websites.
  - Listen to **white noise**, **instrumental music**, or **focus playlists** on platforms like Spotify or YouTube.
- **Create a Dedicated Workspace:**
  - Make your workspace comfortable and organized.
  - Keep only the materials you need for the current task.

---

## **2. Use Time Management Techniques**
- **Pomodoro Technique:**  
  Work for **25 minutes**, then take a **5-minute break**. After 4 cycles, take a longer break (15–30 minutes).  
  *Why it works:* It balances productivity with rest and prevents burnout.
  

### Finishing this conversation

In [37]:
# finishing the conversation
instructions = [{"role": "user", "content": "That's all, thank you!"}]
response=wrapped_chain.invoke({"input": format_instructions(instructions)})
print(response)

You're very welcome! 🌟

If you ever need more tips—whether it's about **gardening**, **staying focused**, **bug control**, or anything else—I'm here to help. Happy gardening and stay sharp!

**Take care and enjoy your green thumb and concentrated moments!** 🌱🧠✨


## Task 4.3: Chatbot using prompt template (LangChain)

In this task, you use the default PromptTemplate that is responsible for the construction of this input. LangChain provides several classes and functions to make constructing and working with prompts easy.

In [38]:
#  prompt for a conversational agent
def format_prompt(actor:str, input:str):
    formatted_prompt: List[str] = []
    if actor == "system":
        prompt_template="""<|begin_of_text|><|start_header_id|>{actor}<|end_header_id|>\n{input}<|eot_id|>"""
    elif actor == "user":
        prompt_template="""<|start_header_id|>{actor}<|end_header_id|>\n{input}<|eot_id|>"""
    else:
        raise ValueError(f"Invalid role: {actor}. Role must be either 'user' or 'system'.")   
    prompt = PromptTemplate.from_template(prompt_template)     
    formatted_prompt.extend(prompt.format(actor=actor,input=input))
    formatted_prompt.extend(["<|start_header_id|>assistant<|end_header_id|>\n"])
    return "".join(formatted_prompt)

In [39]:
# chat user experience
import ipywidgets as ipw
from IPython.display import display, clear_output

class ChatUX:
    """ A chat UX using IPWidgets
    """
    def __init__(self, qa, retrievalChain = False):
        self.qa = qa
        self.name = None
        self.b=None
        self.retrievalChain = retrievalChain
        self.out = ipw.Output()


    def start_chat(self):
        print("Starting chat bot")
        display(self.out)
        self.chat(None)


    def chat(self, _):
        if self.name is None:
            prompt = ""
        else: 
            prompt = self.name.value
        if 'q' == prompt or 'quit' == prompt or 'Q' == prompt:
            with self.out:
                print("Thank you , that was a nice chat !!")
            return
        elif len(prompt) > 0:
            with self.out:
                thinking = ipw.Label(value="Thinking...")
                display(thinking)
                try:
                    if self.retrievalChain:
                        response = self.qa.invoke({"input": prompt})
                        result=response['answer']
                    else:
                        instructions = [{"role": "user", "content": prompt}]
                        #result = self.qa.invoke({'input': format_prompt("user",prompt)}) #, 'history':chat_history})
                        result = self.qa.invoke({"input": format_instructions(instructions)})
                except:
                    result = "No answer"
                thinking.value=""
                print(f"AI:{result}")
                self.name.disabled = True
                self.b.disabled = True
                self.name = None

        if self.name is None:
            with self.out:
                self.name = ipw.Text(description="You:", placeholder='q to quit')
                self.b = ipw.Button(description="Send")
                self.b.on_click(self.chat)
                display(ipw.Box(children=(self.name, self.b)))

Next, start a chat.

In [40]:
# start chat
history = InMemoryChatMessageHistory() #reset chat history
chat = ChatUX(wrapped_chain)
chat.start_chat()

Starting chat bot


Output()

In [41]:
print(history)

Human: <|start_header_id|>user<|end_header_id|>
what do you do <|eot_id|><|start_header_id|>assistant<|end_header_id|>

AI: ### What I Do

As an AI system, my primary purpose is to **assist users by providing helpful, accurate, and reliable information and capabilities** across a wide range of topics and tasks. Here's a breakdown of what I can do:

---

## 📚 **Information & Knowledge Provision**

### 1. **Answer Questions**
- Provide explanations, facts, and insights on **any topic**—science, history, technology, literature, health, etc.
- Clarify complex concepts in simple terms.
- Offer **evidence-based** responses where applicable.

### 2. **Summarize Content**
- Condense long articles, research papers, or documents.
- Highlight key points, themes, or takeaways.

### 3. **Explain How Things Work**
- Break down processes (e.g., how engines work, how code compiles, how ecosystems function).
- Offer step-by-step guides or tutorials.

---

## 🛠️ **Content Creation & Writing Support**

#

## Task 4.4: Chatbot with persona

In this task, Artificial Intelligence(AI) assistant plays the role of a career coach. You can inform the chatbot about its persona (or role) using a system message. Continue to leverage the InMemoryChatMessageHistory class to maintain conversational context.

In [42]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", " You will be acting as a career coach. Your goal is to give career advice to users. For questions that are not career related, don't provide advice. Say, I don't know."),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

history = InMemoryChatMessageHistory() # reset history

chain = prompt | chat_model | StrOutputParser()

wrapped_chain = RunnableWithMessageHistory(
    chain,
    get_history,
    history_messages_key="career_chat_history",
)

response=wrapped_chain.invoke({"input": "What are the career options in AI?"})
print(response)

/opt/conda/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3579: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### **Career Options in AI**

Artificial Intelligence (AI) is a rapidly growing field with diverse opportunities across industries. Below is a comprehensive list of **career options in AI**, categorized by expertise level and specialization:

---

## **1. Entry-Level Roles**
Ideal for beginners or those transitioning into AI.

1. **Data Analyst**
   - **Role**: Collect, process, and visualize data to support AI model development.
   - **Skills**: Excel, SQL, Python, data visualization (Tableau, Power BI).

2. **Data Engineer**
   - **Role**: Build and maintain data pipelines for AI systems.
   - **Skills**: SQL, Python, ETL tools (Apache Spark, Kafka), cloud platforms (AWS, Azure).

3. **Research Assistant (AI/ML)**
   - **Role**: Support academic or industry research projects.
   - **Skills**: Basic Python, machine learning libraries (scikit-learn), academic writing.

---

## **2. Mid-Level Roles**
For professionals with foundational AI/ML knowledge.

### **A. Machine Learning Roles**

In [44]:
response=wrapped_chain.invoke({"input": "What is AI Trainer"})
print(response)

### Understanding the Role of an AI Trainer  

An **AI Trainer** is a professional responsible for preparing and refining datasets and models to improve the performance of artificial intelligence systems. This role is essential in ensuring that AI behaves correctly, makes accurate predictions, and aligns with ethical guidelines.  

#### **Key Responsibilities of an AI Trainer:**  

1. **Data Preparation:**  
   - Collect, clean, and organize large datasets (text, images, audio, etc.).  
   - Remove biases, duplicates, and irrelevant data to ensure high-quality inputs.  
   - Annotate data (labeling images, tagging text, etc.) to provide clear instructions for AI models.  

2. **Model Training & Evaluation:**  
   - Train AI models using frameworks like TensorFlow, PyTorch, or Hugging Face.  
   - Monitor performance metrics (accuracy, precision, recall) and adjust hyperparameters.  
   - Identify weaknesses in the model and iterate to improve results.  

3. **Ethical Oversight:**  
   

In [45]:
print(history)

Human: What are the career options in AI?
AI: ### **Career Options in AI**

Artificial Intelligence (AI) is a rapidly growing field with diverse opportunities across industries. Below is a comprehensive list of **career options in AI**, categorized by expertise level and specialization:

---

## **1. Entry-Level Roles**
Ideal for beginners or those transitioning into AI.

1. **Data Analyst**
   - **Role**: Collect, process, and visualize data to support AI model development.
   - **Skills**: Excel, SQL, Python, data visualization (Tableau, Power BI).

2. **Data Engineer**
   - **Role**: Build and maintain data pipelines for AI systems.
   - **Skills**: SQL, Python, ETL tools (Apache Spark, Kafka), cloud platforms (AWS, Azure).

3. **Research Assistant (AI/ML)**
   - **Role**: Support academic or industry research projects.
   - **Skills**: Basic Python, machine learning libraries (scikit-learn), academic writing.

---

## **2. Mid-Level Roles**
For professionals with foundational AI/ML

Now, ask a question that is not within this persona's specialty. The model should not answer that question and should give a reason for that.

## Task 4.5 Chatbot with Context

In this task, you ask the chatbot to answer questions based on context that was passed to it. You take a CSV file and use the Titan embeddings model to create a vector representing that context. This vector is stored in Facebook AI Similarity Search (FAISS). When the chatbot is asked a question, you will pass this vector back to the chatbot and have it retrieve the answer using the vector.

### Titan embeddings Model

Embeddings represent words, phrases, or any other discrete items as vectors in a continuous vector space. This allows machine learning models to perform mathematical operations on these representations and capture semantic relationships between them.

You use embeddings for the Retrieval-Augmented Generation (RAG) [document search capability](https://labelbox.com/blog/how-vector-similarity-search-works/).

In [46]:
# model configuration
from langchain_aws.embeddings import BedrockEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import PromptTemplate

br_embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v1", client=bedrock_client)

### FAISS as VectorStore

In order to use embeddings for search, you need a store that can efficiently perform vector similarity searches. In this notebook, you use FAISS, which is an in-memory store. To permanently store vectors, you can use Knowledge Bases for Amazon Bedrock, pgVector, Pinecone, Weaviate, or Chroma.

The LangChain VectorStore APIs are available [here](https://python.langchain.com/v0.2/docs/integrations/vectorstores/).

In [60]:
# vector store
from langchain.document_loaders import CSVLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.indexes.vectorstore import VectorStoreIndexWrapper


loader = CSVLoader("../rag_data/Amazon_SageMaker_FAQs.csv") # --- > 219 docs with 400 chars
documents_aws = loader.load() #
print(f"documents:loaded:size={len(documents_aws)}")

docs = CharacterTextSplitter(chunk_size=2000, chunk_overlap=400, separator=",").split_documents(documents_aws)

print(f"Documents:after split and chunking size={len(docs)}")
vectorstore_faiss_aws = None
try:
    
    vectorstore_faiss_aws = FAISS.from_documents(
        documents=docs,
        embedding = br_embeddings, 
        #**k_args
    )

    print(f"vectorstore_faiss_aws:created={vectorstore_faiss_aws}::")

except ValueError as error:
    if  "AccessDeniedException" in str(error):
        print(f"\x1b[41m{error}\
        \nTo troubeshoot this issue please refer to the following resources.\
         \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
         \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")      
        class StopExecution(ValueError):
            def _render_traceback_(self):
                pass
        raise StopExecution        
    else:
        raise error

documents:loaded:size=153
Documents:after split and chunking size=154
vectorstore_faiss_aws:created=<langchain_community.vectorstores.faiss.FAISS object at 0x7ff4b7161a90>::


In [56]:
import time
from langchain_aws import BedrockEmbeddings

class SlowBedrockEmbeddings(BedrockEmbeddings):

    def _embedding_func(self, text, input_type=None):
        time.sleep(1)
        return super()._embedding_func(text, input_type)

br_embeddings = SlowBedrockEmbeddings(
    client=bedrock_client,
    model_id="amazon.titan-embed-text-v2:0"
)

In [57]:
vectorstore_faiss_aws = FAISS.from_documents(
    documents=docs,
    embedding=br_embeddings
)

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [59]:
print(chat_model.model_id)

us.amazon.nova-2-lite-v1:0


### Run a quick low code test 

You can use a Wrapper class provided by LangChain to query the vector database store and return the relevant documents. This runs a QA Chain with all default values.

In [65]:
chat_llm=ChatBedrock(
    model_id="us.amazon.nova-2-lite-v1:0" , 
    client=bedrock_client)
# wrapper store faiss
wrapper_store_faiss = VectorStoreIndexWrapper(vectorstore=vectorstore_faiss_aws)
print(wrapper_store_faiss.query("R in SageMaker", llm=chat_llm))




### **What is Amazon SageMaker?**

Amazon SageMaker is a **fully managed service** designed to **prepare data** and **build, train, and deploy machine learning (ML) models** for **any use case**. It provides **fully managed infrastructure, tools, and workflows**, making it easy for developers and data scientists to build and deploy ML models without managing the underlying infrastructure.

---

### **Is R Supported with Amazon SageMaker?**

**Yes, R is supported with Amazon SageMaker.**

Here’s how R integrates with Amazon SageMaker:

#### **1. R in SageMaker Notebook Instances**
- SageMaker notebook instances come with a **preinstalled R kernel**.
- You can write and execute R code directly in these notebooks.

#### **2. Reticulate Library**
- SageMaker includes the **`reticulate`** library, which provides an **R interface to the Amazon SageMaker Python SDK**.
- With `reticulate`, ML practitioners can:
  - **Build** R models.
  - **Train** models (including using built-in algorithms o

### Chatbot application

For the chatbot, you need context management, history, vector stores, and many other components. You start by building a Retrieval Augmented Generation (RAG) chain that supports context.

This uses the **create_stuff_documents_chain** and **create_retrieval_chain** functions

### Parameters and functions used for RAG

- **Retriever:** You used `VectorStoreRetriever`, which is backed by a `VectorStore`. To retrieve text, there are two search types to choose from: `"similarity"` or `"mmr"`. `search_type="similarity"` uses similarity search in the retriever object, where it selects text chunk vectors that are most similar to the question vector.

- **create_stuff_documents_chain** specifies how retrieved context is fed into a prompt and LLM. Retrieved documents are "stuffed" as context without any summarization or other processing into the prompt.

- **create_retrieval_chain** adds the retrieval step and propagates the retrieved context through the chain, providing it alongside the final answer. 

If the question asked falls outside the scope of the context, the model will reply that it doesn't know the answer.

In [66]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

retriever=vectorstore_faiss_aws.as_retriever()
question_answer_chain = create_stuff_documents_chain(chat_llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "What is sagemaker?"})
print(response) # shows the document chunks consulted to come up with the answer

{'input': 'What is sagemaker?', 'context': [Document(id='d92dffe4-f920-4c89-94c1-9e88de9bd2ad', metadata={'source': '../rag_data/Amazon_SageMaker_FAQs.csv', 'row': 7}, page_content='\ufeffWhat is Amazon SageMaker?: What if I have my own notebook, training, or hosting environment?\nAmazon SageMaker is a fully managed service to prepare data and build, train, and deploy machine learning (ML) models for any use case with fully managed infrastructure, tools, and workflows.: Amazon SageMaker provides a full end-to-end workflow, but you can continue to use your existing tools with SageMaker. You can easily transfer the results of each stage in and out of SageMaker as your business requirements dictate.'), Document(id='6bd203ea-cad2-405e-9c66-0e8ab41753ec', metadata={'source': '../rag_data/Amazon_SageMaker_FAQs.csv', 'row': 12}, page_content='\ufeffWhat is Amazon SageMaker?: What is Amazon SageMaker Studio?\nAmazon SageMaker is a fully managed service to prepare data and build, train, and dep

Next, start a chat

In [64]:
chat = ChatUX(rag_chain, retrievalChain=True)
chat.start_chat()  # Only answers will be shown here, and not the citations

Starting chat bot


Output()


You have utilized Titan LLM to create a conversational interface with following patterns:

- Chatbot (Basic - without context)
- Chatbot using prompt template(Langchain)
- Chatbot with personas
- Chatbot with context

### Try it yourself

- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

### Cleanup

You have completed this notebook. To move to the next part of the lab, do the following:

- Close this notebook file and continue with **Task 5**.